# Natural Language Processing (NLP): Hands-On Lab

This notebook goes with the lecture slides. You will run real Python code to see each NLP idea in action.

**What you will practice:**
1. Getting started with spaCy
2. Tokenization
3. Parts of Speech (POS) and word relationships
4. Stemming
5. Lemmatization
6. Stop words
7. Vocabulary and matching
8. Putting it all together
9. Practice exercises

**How to use this notebook:** Run each code cell from top to bottom. To run a cell, click it and press **Shift + Enter**. Read the comments (the lines that start with `#`). They explain what each line does.

---
## Step 0: Setup

We need two libraries:
- **spaCy** is a modern NLP library. It does many steps at once.
- **NLTK** is a classic NLP library. We use it for stemming.

Run the cell below once. It installs the libraries and downloads a small English model. This may take a minute.

In [ ]:
# The "!" tells the notebook to run a command, not Python code.
# "-q" means "quiet," so it prints less text.
!pip install -q spacy nltk

# Download spaCy's small English model.
# A "model" is a file that already knows a lot about the English language.
!python -m spacy download en_core_web_sm

> **Note:** If you are using Google Colab and you get an error when loading the model below, click **Runtime > Restart session**, then continue from the next cell.

---
## Part 1: Getting Started with spaCy

The `nlp()` function is the heart of spaCy. You give it raw text, and it gives you back a **Doc**. A Doc holds every token plus extra information about each one.

In [ ]:
# Import the spaCy library so we can use it.
import spacy

# Load the small English model and save it in a variable called "nlp".
# From now on, "nlp" is our NLP pipeline.
nlp = spacy.load("en_core_web_sm")

# Let's see which steps are inside the pipeline.
# Each name is one "station" in the assembly line.
print("Pipeline steps:", nlp.pipe_names)

In [ ]:
# Give a sentence to the pipeline.
# The result is a Doc object, which we store in the variable "doc".
doc = nlp("I love learning about AI!")

# A Doc is like a list of tokens. We can loop through it one token at a time.
for token in doc:
    # token.text is the word itself.
    # token.pos_ is the part of speech (the word's job).
    print(token.text, "->", token.pos_)

---
## Part 2: Tokenization

**Tokenization** breaks text into small pieces called **tokens**. Tokens are exact pieces of the original text. Nothing is changed yet.

### 2.1 Why not just split at the spaces?
Let's compare the simple way with spaCy's smarter way.

In [ ]:
text = '"We\'re moving to L.A.!"'

# The simple way: Python's split() cuts the text at every space.
simple_tokens = text.split()
print("Split on spaces:", simple_tokens)

# The smart way: spaCy uses prefix, suffix, infix, and exception rules.
doc = nlp(text)
spacy_tokens = [token.text for token in doc]   # make a list of the token texts
print("spaCy tokens:   ", spacy_tokens)

# Notice how spaCy separated the quote marks, split "We're" into "We" + "'re",
# and kept "L.A." together while splitting off the "!".

### 2.2 The four tokenizer rules in action
- **Prefix:** characters at the start, like `$ ( "`
- **Suffix:** characters at the end, like `km ) , . !`
- **Infix:** characters in between, like `- / ...`
- **Exception:** special cases, like `let's` or `U.S.`

In [ ]:
doc = nlp("It costs $10.50 to ride a taxi in the U.S. for 5km!")

# Print each token with its position number (index).
# enumerate() gives us a counter (i) together with each token.
for i, token in enumerate(doc):
    print(i, token.text)

# Count the tokens with len().
print("\nNumber of tokens:", len(doc))

In [ ]:
# Exceptions: some words are split in a special way.
doc = nlp("Let's visit the U.S. next year. We can't wait!")
print([token.text for token in doc])

# "Let's" -> "Let" + "'s"   and   "can't" -> "ca" + "n't"
# "U.S." stays one token because it is an abbreviation.

### 2.3 Tokens have useful attributes
Every token carries extra facts. Here are a few:
- `is_alpha`: Is it made of letters only?
- `is_punct`: Is it punctuation?
- `like_num`: Does it look like a number?
- `is_stop`: Is it a stop word?

In [ ]:
doc = nlp("It costs $10.50 to ride a taxi in the U.S. for 5km!")

# Print a small table. The ":<8" part adds spaces so the columns line up.
print(f"{'TOKEN':<8}{'ALPHA':<8}{'PUNCT':<8}{'NUMBER':<8}")
for token in doc:
    print(f"{token.text:<8}{str(token.is_alpha):<8}{str(token.is_punct):<8}{str(token.like_num):<8}")

In [ ]:
# Attributes help us FILTER text.
# Example 1: keep only the numbers.
numbers = [token.text for token in doc if token.like_num]
print("Numbers found:", numbers)

# Example 2: remove all punctuation.
no_punct = [token.text for token in doc if not token.is_punct]
print("Without punctuation:", no_punct)

In [ ]:
# A Doc can also be sliced like a Python list.
doc = nlp("Natural Language Processing is fun to learn.")

print("First token:", doc[0])        # index 0 is the first token
print("Last token:", doc[-1])        # index -1 is the last token
print("Tokens 0 to 2:", doc[0:3])    # a slice is called a "Span"

# A Doc also knows where sentences start and end.
doc2 = nlp("This is the first sentence. This is the second one. And a third!")
for sent in doc2.sents:
    print("Sentence:", sent.text)

---
## Part 3: Parts of Speech (POS) and Word Relationships

**POS tagging** labels each token with its grammar job, like NOUN, VERB, or ADJ.

In [ ]:
doc = nlp("The quick brown fox jumps over the lazy dog.")

# token.pos_  = the simple POS tag (like NOUN, VERB)
# spacy.explain() turns a short tag into plain English
for token in doc:
    print(f"{token.text:<8}{token.pos_:<8}{spacy.explain(token.pos_)}")

In [ ]:
# The same word can have different jobs.
# Watch how the tag for "book" changes with context.
doc1 = nlp("I read a book.")
doc2 = nlp("Please book a flight.")

print("Sentence 1:", [(t.text, t.pos_) for t in doc1])
print("Sentence 2:", [(t.text, t.pos_) for t in doc2])

In [ ]:
# Count how many of each POS tag appear in a text.
from collections import Counter   # Counter counts items for us

doc = nlp("My cat sleeps on the soft bed. My dog runs fast in the big park.")
pos_counts = Counter(token.pos_ for token in doc)

for tag, count in pos_counts.most_common():
    print(f"{tag:<7} {count}")

### 3.1 Visualizing word relationships
spaCy's **displaCy** tool draws arrows between words. The arrows show which word describes or depends on another.

In [ ]:
from spacy import displacy

doc = nlp("The quick brown fox jumps over the lazy dog.")

# style="dep" draws the dependency arrows.
# jupyter=True shows the picture inside the notebook.
# "distance" controls the space between words.
displacy.render(doc, style="dep", jupyter=True, options={"distance": 110})

In [ ]:
# We can also print the same relationships as text.
# token.dep_  = the relationship label
# token.head  = the word this token points to
for token in doc:
    print(f"{token.text:<7} --{token.dep_:>6}--> {token.head.text}")

In [ ]:
# displaCy can also highlight named entities (people, places, money, dates).
doc = nlp("Apple is opening a new store in Manila next year for $5 million.")
displacy.render(doc, style="ent", jupyter=True)

---
## Part 4: Stemming

**Stemming** chops off word endings to find a root (the stem). spaCy does **not** have a stemmer, so we use **NLTK**.

We will try two stemmers:
- **Porter Stemmer**: the classic one from 1980
- **Snowball Stemmer**: an improved version, also called "Porter2"

In [ ]:
# Import the two stemmers from NLTK.
from nltk.stem import PorterStemmer, SnowballStemmer

# Create one of each. Snowball needs to know the language.
porter = PorterStemmer()
snowball = SnowballStemmer(language="english")

# Try the stemmer on a family of related words.
words = ["connect", "connected", "connecting", "connection", "connections"]

for word in words:
    print(f"{word:<12} -> {porter.stem(word)}")

In [ ]:
# Stemming makes the vocabulary smaller.
# A "set" keeps only unique items, so we can count different words.
print("Before stemming:", set(words), "->", len(set(words)), "unique words")

stems = [porter.stem(w) for w in words]
print("After stemming: ", set(stems), "->", len(set(stems)), "unique word")

### 4.1 Porter's Phase 1 rules
- `SSES -> SS` (caresses -> caress)
- `IES -> I` (ponies -> poni)
- `SS -> SS` (caress -> caress)
- `S -> (remove)` (cats -> cat)

Only the rule with the **longest matching ending** is used.

In [ ]:
for word in ["caresses", "ponies", "caress", "cats"]:
    print(f"{word:<10} -> {porter.stem(word)}")

# Notice "poni". A stem does NOT need to be a real word.

### 4.2 Porter vs. Snowball

In [ ]:
words = ["running", "runs", "easily", "fairly", "generously", "boating", "boats"]

print(f"{'WORD':<12}{'PORTER':<12}{'SNOWBALL':<12}")
for word in words:
    print(f"{word:<12}{porter.stem(word):<12}{snowball.stem(word):<12}")

# Look at "fairly" and "generously". Snowball gives cleaner stems.

### 4.3 The problems with stemming

In [ ]:
# Problem 1: Over-stemming. Unrelated words get the same stem.
print("universe   ->", porter.stem("universe"))
print("university ->", porter.stem("university"))

# Problem 2: Fake words that are hard to read.
print("happiness  ->", porter.stem("happiness"))
print("studies    ->", porter.stem("studies"))

# Problem 3: Lost meaning. Is an organization a body part?
print("organization ->", porter.stem("organization"))

# Problem 4: Irregular words are not handled.
print("ran   ->", porter.stem("ran"))     # should be "run"
print("geese ->", porter.stem("geese"))   # should be "goose"

---
## Part 5: Lemmatization

**Lemmatization** finds the **dictionary form** of a word, called the **lemma**. It uses a vocabulary and the word's part of speech, so the result is always a real word.

In spaCy, the lemma is stored in `token.lemma_`.

In [ ]:
text = "I am a runner running in a race because I love to run since I ran today"
doc = nlp(text)

print(f"{'TOKEN':<10}{'POS':<8}{'LEMMA':<10}")
for token in doc:
    print(f"{token.text:<10}{token.pos_:<8}{token.lemma_:<10}")

# Key points:
#   am      -> be      (a form of the verb "be")
#   running -> run     and   ran -> run
#   runner  -> runner  (a runner is a person, a NOUN, not an action)

### 5.1 Stemming vs. lemmatization, side by side

In [ ]:
# This helper function compares the two methods for any sentence.
def compare(sentence):
    doc = nlp(sentence)
    print(f"{'WORD':<14}{'STEM':<14}{'LEMMA':<14}")
    for token in doc:
        if token.is_punct:      # skip punctuation to keep the table clean
            continue
        print(f"{token.text:<14}{porter.stem(token.text):<14}{token.lemma_:<14}")
    print()

compare("The geese have big feet.")
compare("She studies hard and organizes her notes.")

### 5.2 Context changes the lemma
The word "meeting" can be a verb (an action) or a noun (an event).

In [ ]:
for sentence in ["I am meeting my friends later.", "The meeting was very long."]:
    doc = nlp(sentence)
    for token in doc:
        if token.text == "meeting":
            print(f"{sentence:<35} POS={token.pos_:<5} lemma={token.lemma_}")

# The stemmer gives "meet" both times, because it ignores context:
print("\nPorter stem of 'meeting':", porter.stem("meeting"))

---
## Part 6: Stop Words

**Stop words** are very common words, like *the, is, in, and*. They add little meaning, so we often remove them.

In [ ]:
# spaCy keeps its stop word list here:
stop_words = nlp.Defaults.stop_words

print("Number of stop words:", len(stop_words))

# sorted() puts them in A-Z order. We show only the first 30.
print(sorted(stop_words)[:30])

In [ ]:
# Check a few words one by one.
for word in ["the", "is", "fox", "over", "not", "python"]:
    print(f"{word:<8} is a stop word? {nlp.vocab[word].is_stop}")

In [ ]:
doc = nlp("The quick brown fox jumps over the lazy dog.")

# Keep a token only if it is NOT a stop word and NOT punctuation.
kept = [token.text for token in doc if not token.is_stop and not token.is_punct]

print("Original:", doc.text)
print("Cleaned: ", " ".join(kept))   # " ".join() glues the words back with spaces

### 6.1 Adding and removing your own stop words

In [ ]:
# ADD a stop word, for example chat slang "btw".
# We must update two places: the list, and the word's flag in the vocabulary.
nlp.Defaults.stop_words.add("btw")
nlp.vocab["btw"].is_stop = True
print("Is 'btw' a stop word now?", nlp.vocab["btw"].is_stop)

# REMOVE a stop word. Here we undo our change by removing "btw" again.
nlp.Defaults.stop_words.remove("btw")
nlp.vocab["btw"].is_stop = False
print("Is 'btw' a stop word now?", nlp.vocab["btw"].is_stop)

### 6.2 Careful! Stop words can change the meaning

In [ ]:
review = "This movie is not good at all."
doc = nlp(review)

# Remove ALL stop words, including "not".
cleaned = [t.text for t in doc if not t.is_stop and not t.is_punct]
print("Original review:", review)
print("After removing stop words:", cleaned)
print("Uh oh! The negative review now looks positive.\n")

# A safer way: keep negative words even though they are stop words.
keep_these = {"not", "no", "never", "n't"}
safer = [t.text for t in doc
         if (not t.is_stop or t.lower_ in keep_these) and not t.is_punct]
print("Safer cleaning:", safer)

---
## Part 7: Vocabulary and Matching

**Rule-based matching** finds phrases that fit a **pattern** that we design. It is like Ctrl+F, but much smarter.

### 7.1 The Matcher: one pattern per spelling
A pattern is a **list of dictionaries**. Each dictionary describes **one token**.

In [ ]:
from spacy.matcher import Matcher

# Create a Matcher that uses the same vocabulary as our nlp pipeline.
matcher = Matcher(nlp.vocab)

# Pattern 1: one token, "solarpower"
pattern1 = [{"LOWER": "solarpower"}]

# Pattern 2: "solar" + any punctuation (like "-") + "power"
pattern2 = [{"LOWER": "solar"}, {"IS_PUNCT": True}, {"LOWER": "power"}]

# Pattern 3: "solar" + "power" as two separate tokens
pattern3 = [{"LOWER": "solar"}, {"LOWER": "power"}]

# Add all three patterns under one name: "SolarPower".
matcher.add("SolarPower", [pattern1, pattern2, pattern3])

text = ("The Solar Power industry keeps growing. "
        "Solar-power panels are cheaper now, and solarpower is popular.")
doc = nlp(text)

# matcher(doc) returns a list of (match_id, start, end)
matches = matcher(doc)
for match_id, start, end in matches:
    rule_name = nlp.vocab.strings[match_id]   # turn the ID back into the name
    print(f"{rule_name}: '{doc[start:end].text}' (tokens {start} to {end})")

### 7.2 Matching with parts of speech
We do not have to type the actual words. We can describe them by their POS tags.

In [ ]:
matcher = Matcher(nlp.vocab)

# Find an adjective followed by a noun.
pattern = [{"POS": "ADJ"}, {"POS": "NOUN"}]
matcher.add("AdjNoun", [pattern])

doc = nlp("The friendly teacher gave us a difficult quiz about smart robots.")

for match_id, start, end in matcher(doc):
    print(doc[start:end].text)

In [ ]:
# Real-world use: find how customers describe products in reviews.
reviews = nlp("The cheap charger stopped working, but the bright screen "
              "and the long battery life are great.")

for match_id, start, end in matcher(reviews):
    print("Description found:", reviews[start:end].text)

### 7.3 Special pattern options
- `"OP": "?"` means the token is optional (0 or 1 times).
- `"OP": "+"` means one or more times.
- `"OP": "*"` means zero or more times.

In [ ]:
matcher = Matcher(nlp.vocab)

# "very" is optional, then an adjective, then a noun.
# This finds "big house" AND "very big house".
pattern = [{"LOWER": "very", "OP": "?"}, {"POS": "ADJ"}, {"POS": "NOUN"}]
matcher.add("DescribedNoun", [pattern])

doc = nlp("They live in a very big house with a small garden.")

# The matcher may find overlapping results, so we use spacy.util.filter_spans
# to keep only the longest match in each place.
spans = [doc[start:end] for _, start, end in matcher(doc)]
for span in spacy.util.filter_spans(spans):
    print(span.text)

### 7.4 The PhraseMatcher: matching a list of terms
When you already have a **list of words or phrases**, the PhraseMatcher is easier.

In [ ]:
from spacy.matcher import PhraseMatcher

# attr="LOWER" means capital letters do not matter.
phrase_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

foods = ["adobo", "sinigang", "halo-halo", "lechon"]

# make_doc() turns each phrase into a Doc so the matcher can use it.
patterns = [nlp.make_doc(food) for food in foods]
phrase_matcher.add("FilipinoFood", patterns)

doc = nlp("We ate Adobo for lunch, Lechon for dinner, and Halo-Halo for dessert.")

for match_id, start, end in phrase_matcher(doc):
    print("Found food:", doc[start:end].text)

---
## Part 8: Putting It All Together

Let's run one movie review through the whole pipeline, step by step.

In [ ]:
review = "The movies weren't boring at all! I loved the actors."
doc = nlp(review)

# Step 1: Tokens
print("1. Tokens:  ", [t.text for t in doc])

# Step 2: POS tags
print("2. POS tags:", [t.pos_ for t in doc])

# Step 3: Lemmas
print("3. Lemmas:  ", [t.lemma_ for t in doc])

# Step 4: Keep key words. Remove stop words and punctuation,
#         but keep negative words so the meaning stays correct.
negatives = {"not", "no", "never", "n't"}
key_words = [t.lemma_ for t in doc
             if (not t.is_stop or t.lower_ in negatives) and not t.is_punct]
print("4. Key words:", key_words)

In [ ]:
# Step 5: A very simple sentiment checker using our key words.
# Real systems use machine learning, but this shows the idea.

positive_words = {"love", "great", "good", "amazing", "fun"}
negative_words = {"boring", "bad", "awful", "slow", "hate"}

def simple_sentiment(text):
    doc = nlp(text)
    score = 0
    flip = False                       # becomes True right after "not"
    for t in doc:
        lemma = t.lemma_.lower()
        if lemma in {"not", "n't", "never", "no"}:
            flip = True                # the next feeling word will be flipped
            continue
        value = 0
        if lemma in positive_words:
            value = 1
        elif lemma in negative_words:
            value = -1
        if value != 0:
            score += -value if flip else value
            flip = False
    if score > 0:
        return f"POSITIVE (score {score})"
    elif score < 0:
        return f"NEGATIVE (score {score})"
    return "NEUTRAL (score 0)"

for r in ["The movies weren't boring at all! I loved the actors.",
          "This movie is not good at all.",
          "The story was slow and boring."]:
    print(f"{r:<55} -> {simple_sentiment(r)}")

---
## Part 9: Practice Exercises

Try these on your own. Write your code where you see `# YOUR CODE HERE`. The answers are at the bottom of the notebook, but try first!

We will use this sentence:
> "The students weren't studying for the tests, but they loved the games!"

In [ ]:
practice = "The students weren't studying for the tests, but they loved the games!"
doc = nlp(practice)

# Exercise 1: Print every token and count how many tokens there are.
# YOUR CODE HERE

In [ ]:
# Exercise 2: Print each token with its POS tag.
# YOUR CODE HERE

In [ ]:
# Exercise 3: Compare the Porter stem and the spaCy lemma
#             for "students", "studying", "tests", and "loved".
# YOUR CODE HERE

In [ ]:
# Exercise 4: Remove stop words and punctuation, but keep "n't" and "not".
# YOUR CODE HERE

In [ ]:
# Exercise 5 (Challenge): Use a Matcher to find every
#                         DETERMINER + NOUN pair (like "the tests").
# Hint: the POS tag for a determiner is "DET".
# YOUR CODE HERE

---
## Answer Key

In [ ]:
practice = "The students weren't studying for the tests, but they loved the games!"
doc = nlp(practice)

# Exercise 1
print("Ex 1:", [t.text for t in doc], "->", len(doc), "tokens\n")

# Exercise 2
print("Ex 2:")
for t in doc:
    print(f"  {t.text:<10}{t.pos_}")

# Exercise 3
print("\nEx 3:")
for t in doc:
    if t.text in ["students", "studying", "tests", "loved"]:
        print(f"  {t.text:<10} stem={porter.stem(t.text):<8} lemma={t.lemma_}")

# Exercise 4
keep = {"not", "n't"}
cleaned = [t.text for t in doc if (not t.is_stop or t.lower_ in keep) and not t.is_punct]
print("\nEx 4:", cleaned)

# Exercise 5
matcher = Matcher(nlp.vocab)
matcher.add("DetNoun", [[{"POS": "DET"}, {"POS": "NOUN"}]])
print("\nEx 5:", [doc[s:e].text for _, s, e in matcher(doc)])

---
## Great Job!

You used real NLP tools to:
- break text into **tokens**,
- label each word's **part of speech**,
- reduce words with **stemming** and **lemmatization**,
- clean text by removing **stop words** carefully,
- and find phrases using **patterns**.

These are the same building blocks used inside search engines, chatbots, and translation apps. Keep experimenting by changing the sentences and running the cells again!